# Phase 25 — Final Documentation & Portfolio Presentation

**Project:** Australian Stock Market Analytics & Return Modeling

## Objective
Collect the outputs produced by earlier phases, generate a reproducible project README and final research report, inventory available figures/tables, and create a limitations register. This notebook does not retrain models or alter earlier results.

**Important:** The report summarizes artifacts found on disk. Missing outputs are explicitly recorded rather than invented. Review the generated documentation before publishing.

## 1. Setup and locate the project

In [1]:
from pathlib import Path
import json
import textwrap
import pandas as pd
import numpy as np
from datetime import datetime

def find_root(start=Path.cwd()):
    required_markers = [
        Path("data/processed/australian_stocks_cleaned.csv"),
        Path("reports/final_model_evaluation/phase23_final_model_evaluation_summary.json"),
        Path("reports/robustness_error_analysis/phase24_robustness_error_analysis_summary.json")
    ]
    for p in [start, *start.parents]:
        if all((p / marker).exists() for marker in required_markers):
            return p
    for p in start.iterdir() if start.exists() else []:
        if p.is_dir() and all((p / marker).exists() for marker in required_markers):
            return p
    return None

PROJECT_ROOT = find_root()
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not locate the project root. Run this notebook from the project folder or notebooks folder. "
        "Expected cleaned data and Phase 23/24 summary files."
    )

DOCS = PROJECT_ROOT / "reports" / "final_documentation"
DOCS.mkdir(parents=True, exist_ok=True)
print("Project root:", PROJECT_ROOT)
print("Documentation output:", DOCS)

Project root: e:\Projects\Australian Stock Market Data Science Project
Documentation output: e:\Projects\Australian Stock Market Data Science Project\reports\final_documentation


## 2. Load Phase 23/24 summaries and available earlier reports

In [2]:
def load_json_if_exists(path):
    if not path.exists():
        return None
    try:
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    except Exception as exc:
        return {"read_error": str(exc), "path": str(path)}

phase23_path = PROJECT_ROOT / "reports/final_model_evaluation/phase23_final_model_evaluation_summary.json"
phase24_path = PROJECT_ROOT / "reports/robustness_error_analysis/phase24_robustness_error_analysis_summary.json"
phase23 = load_json_if_exists(phase23_path)
phase24 = load_json_if_exists(phase24_path)

candidate_summaries = [
    PROJECT_ROOT / "reports/predictive_modeling_strategy/phase7_summary.json",
    PROJECT_ROOT / "reports/market_data_acquisition/phase8_acquisition_summary.json",
    PROJECT_ROOT / "reports/target_engineering/phase9_target_engineering_summary.json",
    PROJECT_ROOT / "reports/predictive_feature_preparation/phase10_feature_preparation_summary.json",
    PROJECT_ROOT / "reports/final_model_evaluation/phase23_final_model_evaluation_summary.json",
    PROJECT_ROOT / "reports/robustness_error_analysis/phase24_robustness_error_analysis_summary.json",
]
earlier_summaries = {str(p.relative_to(PROJECT_ROOT)): load_json_if_exists(p) for p in candidate_summaries if p.exists()}
print("Phase 23 summary loaded:", phase23 is not None)
print("Phase 24 summary loaded:", phase24 is not None)
print("Additional phase summaries found:", list(earlier_summaries))

Phase 23 summary loaded: True
Phase 24 summary loaded: True
Additional phase summaries found: ['reports\\final_model_evaluation\\phase23_final_model_evaluation_summary.json', 'reports\\robustness_error_analysis\\phase24_robustness_error_analysis_summary.json']


## 3. Inventory reports, plots, and tabular artifacts

In [3]:
report_roots = [
    PROJECT_ROOT / "reports",
    PROJECT_ROOT / "data/processed",
]
artifact_rows = []
seen = set()
for base in report_roots:
    if not base.exists():
        continue
    for path in base.rglob("*"):
        if path.is_file() and path not in seen:
            seen.add(path)
            suffix = path.suffix.lower()
            artifact_rows.append({
                "relative_path": str(path.relative_to(PROJECT_ROOT)),
                "extension": suffix,
                "size_kb": round(path.stat().st_size / 1024, 2),
                "artifact_type": (
                    "figure" if suffix in {".png", ".jpg", ".jpeg", ".svg"}
                    else "table" if suffix in {".csv", ".xlsx"}
                    else "report" if suffix in {".md", ".txt", ".json", ".html", ".pdf"}
                    else "other"
                )
            })
artifact_inventory = pd.DataFrame(artifact_rows).sort_values("relative_path")
display(artifact_inventory.head(100))
artifact_inventory.to_csv(DOCS / "artifact_inventory.csv", index=False)
print("Artifacts inventoried:", len(artifact_inventory))

,relative_path,extension,size_kb,artifact_type
163,data\processed\all_model_predictions\all_candi...,.csv,1387.16,table
164,data\processed\audited_features\asx_3m_audited...,.csv,36793.75,table
153,data\processed\australian_stocks_cleaned.csv,.csv,13.39,table
154,data\processed\australian_stocks_feature_engin...,.csv,26.08,table
165,data\processed\corrected_model_predictions\asx...,.csv,809.91,table
...,...,...,...,...
68,reports\final_model_evaluation\test_metrics_by...,.csv,2.49,table
69,reports\final_model_evaluation\test_residual_d...,.png,25.91,figure
70,reports\final_model_evaluation\validation_comp...,.csv,0.75,table
71,reports\final_model_evaluation\validation_mae_...,.png,34.03,figure


Artifacts inventoried: 180


## 4. Extract final model results for documentation

In [4]:
if not isinstance(phase23, dict):
    raise ValueError("Phase 23 summary could not be loaded; complete Phase 23 before finalizing.")

test_metrics = phase23.get("test_metrics", {})
baseline_metrics = phase23.get("test_baseline_metrics", {})
selected_model = phase23.get("selected_model_by_validation_mae", "Not recorded")
target = phase23.get("target", "Not recorded")
features = phase23.get("features", [])

model_comparison = pd.DataFrame([
    {
        "model": test_metrics.get("selected_model", selected_model),
        "role": "Selected model",
        "n": test_metrics.get("n"),
        "MAE": test_metrics.get("MAE"),
        "RMSE": test_metrics.get("RMSE"),
        "R2": test_metrics.get("R2"),
        "mean_prediction_bias": test_metrics.get("mean_prediction_bias")
    },
    {
        "model": baseline_metrics.get("model", "Train+validation mean baseline"),
        "role": "Baseline",
        "n": baseline_metrics.get("n"),
        "MAE": baseline_metrics.get("MAE"),
        "RMSE": baseline_metrics.get("RMSE"),
        "R2": baseline_metrics.get("R2"),
        "mean_prediction_bias": baseline_metrics.get("mean_prediction_bias")
    }
])
display(model_comparison)
model_comparison.to_csv(DOCS / "final_model_test_comparison.csv", index=False)
print("Target:", target)
print("Selected by validation MAE:", selected_model)
print("Feature count:", len(features))

,model,role,n,MAE,RMSE,R2,mean_prediction_bias
0,Ridge Regression,Selected model,9412,0.145430,0.187049,-0.068638,0.042465
1,Train+validation mean baseline,Baseline,9412,0.143784,0.184964,-0.044950,0.038362


Target: forward_return_3m
Selected by validation MAE: Ridge Regression
Feature count: 18


## 5. Create limitations and reproducibility registers

In [5]:
limitations = [
    {
        "topic": "Predictive performance",
        "finding": "The Phase 23 selected model did not beat the mean baseline on held-out test MAE.",
        "implication": "Do not claim that the current predictors provide reliable incremental return-prediction value."
    },
    {
        "topic": "Negative R-squared",
        "finding": "The reported selected-model test R-squared is negative.",
        "implication": "The model performs poorly relative to a test-mean reference in explaining return variation."
    },
    {
        "topic": "Prediction bias",
        "finding": "The selected model's mean predicted return exceeds the actual test-period mean.",
        "implication": "The model overpredicted average returns during this held-out period."
    },
    {
        "topic": "Overlapping targets",
        "finding": "The target spans 63 future observations per ticker, so nearby target windows overlap.",
        "implication": "Row-level observations are dependent; uncertainty should not be calculated as if every row were independent."
    },
    {
        "topic": "Feature provenance",
        "finding": "Earlier phases identified merge-suffixed columns and required manual point-in-time feature checks.",
        "implication": "Verify feature formulas, timestamp alignment, and benchmark availability before claiming a leakage-free experiment."
    },
    {
        "topic": "Historical scope",
        "finding": "Evaluation covers one historical held-out period.",
        "implication": "Results do not guarantee future performance and are not investment advice."
    }
]
limitations_df = pd.DataFrame(limitations)
display(limitations_df)
limitations_df.to_csv(DOCS / "limitations_register.csv", index=False)

repro_steps = [
    "Use the same Python environment and record package versions.",
    "Keep the original workbook unchanged and retain the generated raw/processed datasets.",
    "Run notebooks in phase order, beginning with data understanding and cleaning.",
    "Confirm all output paths and dataset versions before running downstream notebooks.",
    "Inspect each phase summary and audit warnings rather than relying only on successful cell execution.",
    "For final evaluation, select models on validation data and use the test set only for the final report.",
    "Do not claim investment profitability from historical regression metrics."
]
(DOCS / "reproducibility_checklist.md").write_text(
    "# Reproducibility Checklist\n\n" + "\n".join(f"- [ ] {s}" for s in repro_steps),
    encoding="utf-8"
)

,topic,finding,implication
0,Predictive performance,The Phase 23 selected model did not beat the m...,Do not claim that the current predictors provi...
1,Negative R-squared,The reported selected-model test R-squared is ...,The model performs poorly relative to a test-m...
2,Prediction bias,The selected model's mean predicted return exc...,The model overpredicted average returns during...
3,Overlapping targets,The target spans 63 future observations per ti...,Row-level observations are dependent; uncertai...
4,Feature provenance,Earlier phases identified merge-suffixed colum...,"Verify feature formulas, timestamp alignment, ..."
5,Historical scope,Evaluation covers one historical held-out period.,Results do not guarantee future performance an...


656

## 6. Generate final research report

In [6]:
def fmt_num(value, digits=6):
    try:
        return f"{float(value):.{digits}f}"
    except (TypeError, ValueError):
        return "Not available"

tm = phase23.get("test_metrics", {})
bm = phase23.get("test_baseline_metrics", {})
test_rows = tm.get("n", "Not recorded")
actual_mean = tm.get("actual_mean")
pred_mean = tm.get("predicted_mean")
mae = tm.get("MAE")
baseline_mae = bm.get("MAE")
r2 = tm.get("R2")
beats = phase23.get("test_beats_baseline_mae")

report = f'''# Australian Stock Market Analytics & Return Modeling

## Executive summary

This project builds an end-to-end workflow for analyzing Australian stock data, engineering historical price features, studying company groups, and evaluating a three-month forward-return regression experiment.

The final predictive target is `{target}`, defined over 63 future observations per ticker. The Phase 23 model was selected using validation MAE and evaluated on a held-out chronological test period.

## Final model evaluation

- Model selected on validation MAE: **{selected_model}**
- Test observations: **{test_rows}**
- Number of predictors: **{len(features)}**
- Test MAE: **{fmt_num(mae)}**
- Baseline test MAE: **{fmt_num(baseline_mae)}**
- Test RMSE: **{fmt_num(tm.get("RMSE"))}**
- Test R-squared: **{fmt_num(r2)}**
- Actual mean test return: **{fmt_num(actual_mean)}**
- Mean predicted test return: **{fmt_num(pred_mean)}**
- Did the model beat the baseline on MAE? **{beats}**

### Interpretation

The selected model did **not** beat the mean baseline on held-out test MAE. The test R-squared is negative, and the mean prediction is higher than the realized mean return in the test period. Therefore, the current experiment does not establish useful incremental predictive performance over the baseline.

## Workflow

The project phases cover:
1. Data understanding and workbook inspection.
2. Data cleaning and quality audits.
3. Financial feature engineering.
4. Exploratory data analysis.
5. Statistical analysis.
6. Exploratory company segmentation.
7. Predictive-task feasibility review.
8. Historical market-data acquisition.
9. Forward-return target engineering.
10. Predictive feature preparation.
11–23. Model development, validation, feature audits, and final held-out evaluation.
24. Robustness checks and error analysis.
25. Final documentation and portfolio packaging.

## Methodology overview

The main modeling target is a three-month forward return based on 63 future price observations per ticker. The workflow uses chronological train, validation, and test periods. Model selection is based on validation MAE; the final selected model is refitted on train plus validation before test evaluation.

The selected feature list contains {len(features)} predictors, primarily momentum, volatility, price-to-moving-average ratios, benchmark returns, and relative momentum.

## Limitations

- The current model does not outperform the mean baseline on the held-out test MAE.
- Negative test R-squared indicates weak out-of-sample explanatory performance.
- Forward-return windows overlap, so observations are not independent.
- Feature-name screening alone cannot prove that all features were available at prediction time; source formulas and benchmark alignment require review.
- Performance on one historical period is not evidence of future profitability.
- This project is for educational and research purposes, not investment advice.

## Recommended future work

- Verify point-in-time feature provenance and benchmark date alignment.
- Use walk-forward validation with purging/embargo for overlapping labels.
- Establish a documented baseline protocol before testing new features.
- If further modeling is attempted, reserve a new untouched future period for final evaluation.
- Explore ranking or classification targets only if they are defined in advance and validated without leakage.

## Reproducibility

See `artifact_inventory.csv`, `limitations_register.csv`, and `reproducibility_checklist.md` in the `reports/final_documentation/` directory. Preserve the phase outputs and record package versions and data retrieval dates before publishing.

---
**Disclaimer:** Historical research only. This report is not financial or investment advice.
'''
(DOCS / "final_research_report.md").write_text(report, encoding="utf-8")
print("Saved:", DOCS / "final_research_report.md")
print(report[:5000])

Saved: e:\Projects\Australian Stock Market Data Science Project\reports\final_documentation\final_research_report.md
# Australian Stock Market Analytics & Return Modeling

## Executive summary

This project builds an end-to-end workflow for analyzing Australian stock data, engineering historical price features, studying company groups, and evaluating a three-month forward-return regression experiment.

The final predictive target is `forward_return_3m`, defined over 63 future observations per ticker. The Phase 23 model was selected using validation MAE and evaluated on a held-out chronological test period.

## Final model evaluation

- Model selected on validation MAE: **Ridge Regression**
- Test observations: **9412**
- Number of predictors: **18**
- Test MAE: **0.145430**
- Baseline test MAE: **0.143784**
- Test RMSE: **0.187049**
- Test R-squared: **-0.068638**
- Actual mean test return: **-0.009888**
- Mean predicted test return: **0.032577**
- Did the model beat the baseline on MA

## 7. Generate a portfolio README

In [7]:
readme = f'''# Australian Stock Market Analytics & Return Modeling

An end-to-end Data Science project analyzing Australian listed companies, engineering market features, exploring company groups, and evaluating a three-month forward-return regression task.

## Project goals

- Inspect and clean company fundamentals from the original workbook.
- Engineer financial ratios and historical price-based features.
- Perform exploratory and statistical analysis.
- Explore company segmentation.
- Acquire historical market data and construct forward-return targets.
- Compare regression models using chronological validation.
- Evaluate the final selected model against a simple baseline.
- Document limitations and reproducibility.

## Final result

- **Target:** `{target}`
- **Selected model by validation MAE:** {selected_model}
- **Predictors:** {len(features)}
- **Test observations:** {test_rows}
- **Selected model test MAE:** {fmt_num(mae)}
- **Baseline test MAE:** {fmt_num(baseline_mae)}
- **Test R-squared:** {fmt_num(r2)}
- **Beat baseline on test MAE:** {beats}

**Interpretation:** The selected model did not beat the mean baseline on test MAE. This project reports that outcome transparently and does not claim reliable return-prediction ability.

## Main tools

- Python
- pandas and NumPy
- Matplotlib
- scikit-learn
- SciPy and statsmodels (statistical analysis)
- yfinance (historical market data)
- Jupyter Notebook

## Suggested project structure

```text
.
├── data/
│   ├── raw/
│   ├── metadata/
│   └── processed/
├── notebooks/
├── reports/
│   ├── eda/
│   ├── statistical_analysis/
│   ├── company_segmentation/
│   ├── final_model_evaluation/
│   ├── robustness_error_analysis/
│   └── final_documentation/
└── README.md
```

## How to reproduce

1. Create/activate the Python environment used for the project.
2. Install required packages based on the notebook imports.
3. Place the original workbook in the expected project location.
4. Run notebooks in phase order and inspect each phase's output and warnings.
5. Confirm historical market-data retrieval and output files are present.
6. Review the final report and audit registers.

## Important limitations

- Historical evaluation only; not investment advice.
- The selected model did not beat the baseline on held-out test MAE.
- Forward-return labels overlap over 63 observations.
- Feature provenance and benchmark alignment require manual verification.
- One held-out period cannot establish future performance.

## Reports

See `reports/final_documentation/final_research_report.md` for the final research report and `artifact_inventory.csv` for an inventory of available outputs.
'''
(DOCS / "portfolio_README.md").write_text(readme, encoding="utf-8")
print("Saved:", DOCS / "portfolio_README.md")

Saved: e:\Projects\Australian Stock Market Data Science Project\reports\final_documentation\portfolio_README.md


## 8. Save final project summary

In [8]:
summary = {
    "phase": 25,
    "project_root": str(PROJECT_ROOT),
    "target": target,
    "selected_model_by_validation_mae": selected_model,
    "n_features": len(features),
    "test_rows": test_rows,
    "test_mae": mae,
    "baseline_test_mae": baseline_mae,
    "test_r2": r2,
    "test_beats_baseline_mae": beats,
    "artifact_count": int(len(artifact_inventory)),
    "outputs": [
        str((DOCS / "final_research_report.md").relative_to(PROJECT_ROOT)),
        str((DOCS / "portfolio_README.md").relative_to(PROJECT_ROOT)),
        str((DOCS / "artifact_inventory.csv").relative_to(PROJECT_ROOT)),
        str((DOCS / "limitations_register.csv").relative_to(PROJECT_ROOT)),
        str((DOCS / "reproducibility_checklist.md").relative_to(PROJECT_ROOT)),
        str((DOCS / "final_model_test_comparison.csv").relative_to(PROJECT_ROOT))
    ],
    "warning": "Review generated report, artifact inventory, and manual feature-provenance checks before public release."
}
with open(DOCS / "phase25_final_documentation_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, default=str)
print(json.dumps(summary, indent=2, default=str))

{
  "phase": 25,
  "project_root": "e:\\Projects\\Australian Stock Market Data Science Project",
  "target": "forward_return_3m",
  "selected_model_by_validation_mae": "Ridge Regression",
  "n_features": 18,
  "test_rows": 9412,
  "test_mae": 0.14542968580413784,
  "baseline_test_mae": 0.14378448609198502,
  "test_r2": -0.06863781600094598,
  "test_beats_baseline_mae": false,
  "artifact_count": 180,
  "outputs": [
    "reports\\final_documentation\\final_research_report.md",
    "reports\\final_documentation\\portfolio_README.md",
    "reports\\final_documentation\\artifact_inventory.csv",
    "reports\\final_documentation\\limitations_register.csv",
    "reports\\final_documentation\\reproducibility_checklist.md",
    "reports\\final_documentation\\final_model_test_comparison.csv"
  ],
  "warning": "Review generated report, artifact inventory, and manual feature-provenance checks before public release."
}


## 9. Completion checklist

- Read `final_research_report.md` and `portfolio_README.md`.
- Confirm the project structure and package list match the actual environment.
- Check the artifact inventory for missing expected outputs.
- Manually review feature provenance and benchmark alignment.
- Do not claim that the model beats the baseline.
- Remove any private paths or sensitive data before publishing the repository.